<a href="https://colab.research.google.com/github/lucashobbs17/Geostorm/blob/main/notebooks/01_flare_cme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
   !pip install -q "sunpy[net]"

   from google.colab import drive
   drive.mount('/content/drive')

   import os
   import pandas as pd
   import numpy as np
   import matplotlib.pyplot as plt

   DATA = '/content/drive/MyDrive/geostorm/data'
   os.makedirs(DATA, exist_ok=True)
   print("Setup done")

Mounted at /content/drive
Setup done


   ## 2. Get flares (GOES via HEK)

In [4]:
from sunpy.net import Fido, attrs as a

path = f'{DATA}/flares_2011.csv'

if os.path.exists(path):
    flares = pd.read_csv(path)
    print("Loaded from Drive")
else:
    res = Fido.search(
        a.Time("2011-01-01", "2011-12-31"),
        a.hek.EventType("FL"),
        a.hek.FL.GOESCls > "M1.0",
        a.hek.OBS.Observatory == "GOES",
    )
    cols = ["event_starttime", "event_peaktime", "event_endtime",
            "fl_goescls", "hgs_x", "hgs_y"]
    flares = res["hek"][cols].to_pandas()
    flares.to_csv(path, index=False)
    print("Downloaded and saved")

flares.head()

Loaded from Drive


,event_starttime,event_peaktime,event_endtime,fl_goescls,hgs_x,hgs_y
0,2011-01-28 00:44:00,2011-01-28 01:03:00,2011-01-28 01:10:00,M1.3,0,0
1,2011-02-09 01:23:00,2011-02-09 01:31:00,2011-02-09 01:35:00,M1.9,72,17
2,2011-02-13 17:28:00,2011-02-13 17:38:00,2011-02-13 17:47:00,M6.6,-4,-20
3,2011-02-14 17:20:00,2011-02-14 17:26:00,2011-02-14 17:32:00,M2.2,18,56
4,2011-02-15 01:44:00,2011-02-15 01:56:00,2011-02-15 02:06:00,X2.2,0,0


In [5]:
print(flares.shape)
flares["fl_goescls"].value_counts()

(109, 6)


,count
fl_goescls,
M1.1,12
M1.2,10
M1.3,9
M1.4,8
M1.9,5
M1.6,5
M1.5,5
M1.8,4
M1.7,4


In [6]:
missing = (flares.hgs_x == 0) & (flares.hgs_y == 0)
print("Missing locations:", missing.sum())

dupes = flares[flares.duplicated("event_peaktime", keep=False)]
print("Duplicate rows:", len(dupes))
dupes

Missing locations: 44
Duplicate rows: 2


,event_starttime,event_peaktime,event_endtime,fl_goescls,hgs_x,hgs_y
52,2011-09-06 22:12:00,2011-09-06 22:20:00,2011-09-06 22:24:00,X2.1,18,14
53,2011-09-06 22:12:00,2011-09-06 22:20:00,2011-09-06 22:24:00,X2.1,18,14


In [7]:
flares.loc[missing, ["hgs_x", "hgs_y"]] = np.nan
flares = flares.drop_duplicates("event_peaktime").reset_index(drop=True)
print(flares.shape)

(108, 6)


In [8]:
for c in ["event_starttime", "event_peaktime", "event_endtime"]:
    flares[c] = pd.to_datetime(flares[c])

flares = flares.sort_values("event_peaktime").reset_index(drop=True)

In [9]:
path_ssw = f'{DATA}/ssw_locations_2011.csv'

if os.path.exists(path_ssw):
    ssw = pd.read_csv(path_ssw)
else:
    res2 = Fido.search(
        a.Time("2011-01-01", "2011-12-31"),
        a.hek.EventType("FL"),
        a.hek.FRM.Name == "SSW Latest Events",
    )
    ssw = res2["hek"][["event_peaktime", "hgs_x", "hgs_y"]].to_pandas()
    ssw.to_csv(path_ssw, index=False)

ssw["event_peaktime"] = pd.to_datetime(ssw["event_peaktime"])
ssw = ssw.sort_values("event_peaktime").reset_index(drop=True)
print(ssw.shape)

(2546, 3)


In [10]:
merged = pd.merge_asof(
    flares,
    ssw.rename(columns={"hgs_x": "ssw_x", "hgs_y": "ssw_y"}),
    on="event_peaktime",
    direction="nearest",
    tolerance=pd.Timedelta("10min"),
)

flares["hgs_x"] = flares["hgs_x"].fillna(merged["ssw_x"])
flares["hgs_y"] = flares["hgs_y"].fillna(merged["ssw_y"])
print("Still missing:", flares["hgs_x"].isna().sum())

Still missing: 7


In [12]:
flares[flares.event_peaktime.dt.date == pd.Timestamp("2011-02-15").date()]
print("Zero locations:", ((flares.hgs_x == 0) & (flares.hgs_y == 0)).sum())
zero = (flares.hgs_x == 0) & (flares.hgs_y == 0)
flares.loc[zero, ["hgs_x", "hgs_y"]] = np.nan

flares = flares.dropna(subset=["hgs_x", "hgs_y"]).reset_index(drop=True)
flares.to_csv(f'{DATA}/flares_2011_clean.csv', index=False)
print(flares.shape)

Zero locations: 1
(100, 6)
